In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt

def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test
    
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
 
def Linear(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   
    
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
def featureImportance(indep_X, dep_Y):

    featurelist = []

    from sklearn.linear_model import LogisticRegression
    lin = LogisticRegression(max_iter=1000)

    from sklearn.svm import SVC
    SVRl = SVC(kernel='linear')

    from sklearn.tree import DecisionTreeClassifier
    dec = DecisionTreeClassifier(random_state=0)

    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=100, random_state=0)

    modellist = [lin, SVRl, dec, rf]

    for i in modellist:

        print(i)

        i.fit(indep_X, dep_Y)

        if hasattr(i, 'feature_importances_'):
            importance = i.feature_importances_
        else:
            importance = np.abs(i.coef_).flatten()

        featurelist.append(importance)

    return featurelist

In [3]:
def featureImportance_regression(acclin, accsvml, accsvmnl, accdes, accrf):

    featuredataframe = pd.DataFrame(
        [
            acclin,
            accsvml,
            accsvmnl,
            accdes,
            accrf
        ],
        index=['Linear', 'SVM Linear', 'SVM Nonlinear',
               'DecisionTree', 'RandomForest'],
        columns=['Linear_Features', 'SVC_Features',
                 'DecisionTree_Features', 'RandomForest_Features']
    )

    return featuredataframe

In [4]:
dataset1 = pd.read_csv("prep.csv", index_col=None)

df2 = pd.get_dummies(dataset1, drop_first=True)

indep_X = df2.drop('classification_yes', axis=1)
dep_Y = df2['classification_yes']


In [5]:
featurelist = featureImportance(indep_X, dep_Y)

acclin = []
accsvml = []
accsvmnl = []
accdes = []
accrf = []

for i in featurelist:

    # Get important feature names
    important_features = indep_X.columns[
        np.argsort(i)[::-1][:10]
    ]

    # Get original dataset columns
    selected_X = indep_X[important_features]

    X_train, X_test, y_train, y_test = split_scalar(selected_X, dep_Y)

    r2_lin = Linear(X_train, y_train, X_test)
    acclin.append(r2_lin)

    r2_sl = svm_linear(X_train, y_train, X_test)
    accsvml.append(r2_sl)

    r2_NL = svm_NL(X_train, y_train, X_test)
    accsvmnl.append(r2_NL)

    r2_d = Decision(X_train, y_train, X_test)
    accdes.append(r2_d)

    r2_r = random(X_train, y_train, X_test)
    accrf.append(r2_r)

result = featureImportance_regression( acclin, accsvml, accsvmnl, accdes, accrf)
result

LogisticRegression(max_iter=1000)


C:\Anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


SVC(kernel='linear')
DecisionTreeClassifier(random_state=0)
RandomForestClassifier(random_state=0)


,Linear_Features,SVC_Features,DecisionTree_Features,RandomForest_Features
Linear,0.711640,0.703628,0.696777,0.663173
SVM Linear,0.684787,0.655058,0.653055,0.617880
SVM Nonlinear,0.937867,0.935762,0.932024,0.935645
DecisionTree,0.869792,0.826389,0.782986,0.782986
RandomForest,0.871962,0.895833,0.925781,0.919271
